# Latent DiT world state — VAE quality, predictive quality, and where position lives

**What this notebook establishes**, before any editing question is asked of the latent DiT:
1. the frozen VAE reconstructs observations well and its latent keeps the world state;
2. the latent DiT predicts as well as the pixel-space models (mean mode **and** multi-step sampling);
3. multi-step Euler sampling is stable autoregressively (needed for the gradient-steering work);
4. where position information sits in the (residual point × diffusion time) grid.

**Data / model provenance.** Models `0_latent_dit_z16_w4` = **Latent DiT · z=16 · window 4** (frozen `vae_z16`
+ d256/4-layer DiT core over 16-d latents) and `1_latent_dit_z16_w2` = **Latent DiT · z=16 · window 2**; VAEs
`vae_z16` / `vae_z8`. All rows copied from `LATENT_DIT_RUNS.md`. Dataset `datasets/4_fixed_refl_inview` test
split. Probes: `pim.extractors.fit_readability_probes` (linear lstsq + 2×256 ReLU MLP, 80/20 by-**sequence**
split, `STD_EPOCHS`=300 after the 2026-08-11 undertraining fix).

## Definitions

| term | definition | units | better |
|---|---|---|---|
| **normalised latent z** | `VAE.encode_normalized(obs)` = posterior mean ÷ `latent_scale`; the space the DiT actually runs in (≈ unit scale) | — | — |
| **VAE recon RMSE (noisy / clean)** | RMSE of `decode(encode(obs))` against the noisy input it was trained on / against the simulator's clean render. A tight latent partially denoises, so the clean number is usually better and is the "did the code keep the world state" metric | intensity | ↓ |
| **noise floor** | RMSE(noisy obs, clean render) on this dataset = **0.1539**. The reference scale for every reconstruction number here | intensity | — |
| **decoded next-step RMSE** | RMSE between the model's decoded one-step prediction and the target frame (reported vs both noisy and clean), teacher-forced | intensity | ↓ |
| **VAE floor** | the same metric for `decode(encode(target))` — the best any latent model could do; **every decoded prediction number must be read against it** | intensity | — |
| **residual point ℓ** | DiT-core residual stream via `resid_sink`: 0 = token embedding, 1–3 = input to blocks 2–4, 4 = final-block output (velocity-head input) | — | — |
| **diffusion time τ** | noise level of the prediction slot's iterate when the state is read; states collected from the **actual** fresh-noise Euler sampler (τ=1 pure noise → τ=0 finished sample) | — | — |
| **probe R²** | held-out position R² (both objects, 4 dims) of a probe fit independently at that grid cell | — | ↑ |
| **rollout modes** | `mean` (deterministic conditional-mean readout) · `sample_fresh` (8-step Euler ODE from per-sample fresh noise, the honest generative mode; `sample` reuses one fixed noise vector and is **not** for rollouts) | — | — |

Observation-space errors are scored against the **clean** render unless explicitly labelled "vs noisy".

In [ ]:
# [1] Setup + §1 VAE quality: reconstruction against both references, and what the latent retains.
import os, sys
sys.path.insert(0, "../../../..")
import numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.world_models.latent_dit import LatentDiTModel  # noqa: F401 (loader dispatch)
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, N_PROBE, N_EVAL = 2, 500, 512
OUT = "/tmp/latent_dit_world_state"; os.makedirs(OUT, exist_ok=True)

MODELS = {
    "Latent DiT · z=16 · window 4": "../../../../runs/latent_dit/0_latent_dit_z16_w4/best_model.pt",
    "Latent DiT · z=16 · window 2": "../../../../runs/latent_dit/1_latent_dit_z16_w2/best_model.pt",
    "Latent DiT · z=8 · window 4":  "../../../../runs/latent_dit/2_latent_dit_z8_w4/best_model.pt",
}
PRIMARY = "Latent DiT · z=16 · window 4"
loaded = {k: load_checkpoint(v, device=DEVICE) for k, v in MODELS.items()}
model, info = loaded[PRIMARY]

bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ, require_edits=False)
test = bundle.test
obs_t = torch.from_numpy(test.obs[:N_EVAL]).float().to(DEVICE)
clean_t = torch.from_numpy(test.clean_obs[:N_EVAL]).float().to(DEVICE)
noise_floor = float(((obs_t - clean_t) ** 2).mean().sqrt())
print(f"primary: {PRIMARY} | epoch {info.epoch} | z={model.latent_dim} window={model.window} "
      f"d_model={model.core.cfg.d_model} | device={DEVICE}")
print(f"dataset noise floor RMSE(noisy, clean) = {noise_floor:.4f}")

rows = []
for label, (m, _) in loaded.items():
    with torch.no_grad():
        rec = m.decode_latent(m.encode(obs_t))
    rows.append((label, m.latent_dim,
                 float(((rec - obs_t) ** 2).mean().sqrt()),
                 float(((rec - clean_t) ** 2).mean().sqrt())))
display(Markdown("**§1 — frozen VAE reconstruction on the test split** (N=%d sequences; the noise floor "
                 "RMSE(noisy, clean) is **%.4f** — a reconstruction closer to *clean* than that is denoising, "
                 "not error)\n\n| model (its VAE) | z | recon RMSE vs noisy | recon RMSE vs clean |\n"
                 "|---|---|---|---|\n" % (N_EVAL, noise_floor)
                 + "\n".join(f"| {l} | {z} | {rn:.4f} | {rc:.4f} |" for l, z, rn, rc in rows)))

# what the latent retains, vs the raw observation it replaces
P = test.positions[:N_PROBE, :, :N_OBJ].reshape(N_PROBE, -1, N_OBJ * 2).astype(np.float32)
vis = test.is_visible[:N_PROBE, :, :N_OBJ].all(axis=2)
with torch.no_grad():
    z_probe = model.encode(torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)).cpu().numpy()
r_lat = fit_readability_probes(z_probe, P, mask=vis, device=DEVICE)
r_raw = fit_readability_probes(test.obs[:N_PROBE].astype(np.float32), P, mask=vis, device=DEVICE)
display(Markdown("**§1b — single-frame position readout: the 16-d code vs the 128-d observation it replaces** "
                 "(the comparison is the point; single-frame position readout is intrinsically limited)\n\n"
                 "| source | linear R² | MLP R² |\n|---|---|---|\n"
                 f"| normalised VAE latent (z={model.latent_dim}) | {r_lat['linear_r2']:.3f} | {r_lat['mlp_r2']:.3f} |\n"
                 f"| raw 128-d observation | {r_raw['linear_r2']:.3f} | {r_raw['mlp_r2']:.3f} |"))

In [ ]:
# [2] Fig 1 — VAE reconstruction in observation space (does the code keep the objects?).
SHOW_S, SHOW_T = [0, 1], [8, 20, 32]
x_axis = np.arange(test.obs_res)
with torch.no_grad():
    rec_all = model.decode_latent(model.encode(obs_t)).cpu().numpy()
fig, axes = plt.subplots(len(SHOW_S), len(SHOW_T), figsize=(4.4 * len(SHOW_T), 2.7 * len(SHOW_S)),
                         sharex=True, sharey=True)
for r, s in enumerate(SHOW_S):
    for c, t in enumerate(SHOW_T):
        ax = axes[r, c]
        ax.plot(x_axis, test.obs[s, t], color="#999999", lw=1.0, label="noisy observation (VAE input)")
        ax.plot(x_axis, test.clean_obs[s, t], color="#009E73", lw=1.4, ls="--", label="clean render")
        ax.plot(x_axis, rec_all[s, t], color="#D55E00", lw=1.4, label="VAE reconstruction")
        ax.set_ylim(-0.05, 1.1)
        if r == 0: ax.set_title(f"frame t={t}", fontsize=10)
        if c == 0: ax.set_ylabel(f"test seq {s}\nintensity", fontsize=9)
        if r == len(SHOW_S) - 1: ax.set_xlabel("ray")
        style_ax(ax)
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"Fig 1 — frozen VAE reconstruction (z={model.latent_dim}); the 128-ray scan compressed to "
             f"{model.latent_dim} numbers and decoded back", y=1.05, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"{OUT}/fig1_vae_recon.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [3] §2 Predictive quality: decoded next-step error vs the VAE floor and the pixel-space DiT.
#     The pixel DiT is RE-EVALUATED here on the same sequences (not cited) so both the noisy-target and the
#     clean-target columns are computed identically for both architectures.
with torch.no_grad():
    vae_floor_pred = model.decode_latent(model.encode(obs_t[:, 1:]))
floor_noisy = float(((vae_floor_pred - obs_t[:, 1:]) ** 2).mean())
floor_clean = float(((vae_floor_pred - clean_t[:, 1:]) ** 2).mean())

rows = []
for label, (m, inf) in loaded.items():
    m.predict_mode = "mean"
    with torch.no_grad():
        pred, _ = m(obs_t)
    rows.append((label, float(((pred - obs_t[:, 1:]) ** 2).mean()),
                 float(((pred - clean_t[:, 1:]) ** 2).mean()), f"ep {inf.epoch}"))

PIXEL_REFS = {
    "DiT concat · d256 · window 4 (pixel space)": "../../../../runs/dit/9_dset4_dit_w4_d256/best_model.pt",
    "DiT concat · d256 · window 2 (pixel space)": "../../../../runs/dit/8_dset4_dit_w2_d256/best_model.pt",
}
for label, path in PIXEL_REFS.items():
    pm, pinf = load_checkpoint(path, device=DEVICE)
    pm.predict_mode = "mean"
    with torch.no_grad():
        pred, _ = pm(obs_t)
    rows.append((label, float(((pred - obs_t[:, 1:]) ** 2).mean()),
                 float(((pred - clean_t[:, 1:]) ** 2).mean()), f"ep {pinf.epoch}"))

body = "\n".join(f"| {l} | {mn:.5f} | {np.sqrt(mn):.4f} | {mc:.5f} | {np.sqrt(mc):.4f} | {ep} |"
                 for l, mn, mc, ep in rows)
body += (f"\n| _VAE floor (encode→decode the target frame)_ | {floor_noisy:.5f} | {np.sqrt(floor_noisy):.4f} "
         f"| {floor_clean:.5f} | {np.sqrt(floor_clean):.4f} | — |")
body += "\n| _GRU · H=256 (cited, vs noisy only)_ | 0.02362 | 0.1537 | — | — | — |"
display(Markdown("**§2 — decoded next-step prediction quality** (teacher-forced, N=%d test sequences; all rows "
                 "except the GRU computed here on identical data). The **VAE floor** is what a latent model "
                 "would score with perfect dynamics — read the latent rows against it. The **vs-clean** columns "
                 "are the fairer cross-architecture structure comparison, since they do not reward reproducing "
                 "observation noise.\n\n"
                 "| model | MSE vs noisy | RMSE vs noisy | MSE vs clean | RMSE vs clean | ckpt |\n"
                 "|---|---|---|---|---|---|\n" % N_EVAL + body))
print(f"headline: {PRIMARY} decoded MSE vs noisy {rows[0][1]:.5f} (VAE floor {floor_noisy:.5f}, "
      f"excess {rows[0][1]-floor_noisy:+.5f}) | vs clean {rows[0][2]:.5f} "
      f"vs pixel DiT W4 {rows[3][2]:.5f}")

In [ ]:
# [4] §3 Sampled generative rollouts — is multi-step Euler sampling stable autoregressively?
#     Uses the model API's "sample_fresh" mode (per-sample fresh start noise, seeded via model.noise_gen);
#     the fixed-noise "sample" mode collapses when iterated (2026-08-11 diagnosis, see LATENT_DIT_RUNS.md).
EF, K_ROLL, N_ROLL = 20, 15, 64
obs_roll = obs_t[:N_ROLL]
gt_body = test.clean_obs[:N_ROLL, EF:EF + K_ROLL]

def free_run(m, mode, seed=None, k=K_ROLL):
    """Teacher-force noisy obs[0..EF-1] then k-step free-run (feedback stays in latent space)."""
    m.predict_mode = mode
    m.noise_gen = torch.Generator().manual_seed(seed) if seed is not None else None
    state = None
    with torch.no_grad():
        for t in range(EF):
            _, state = m.step(obs_roll[:, t], state)
        out, z = [], m.core.decode(m._to_core(state))
        for _ in range(k):
            out.append(m.decode_latent(z).cpu().numpy())
            z, state = m.step_latent(z, state)
    m.predict_mode, m.noise_gen = "mean", None
    return np.stack(out, axis=1)

ROLLS = {}
for label, (m, _) in loaded.items():
    ROLLS[label] = {"mean": free_run(m, "mean"),
                    "sample_fresh A": free_run(m, "sample_fresh", seed=101),
                    "sample_fresh B": free_run(m, "sample_fresh", seed=202)}
rows = []
for label, rs in ROLLS.items():
    for mode, r in rs.items():
        rows.append(f"| {label} | {mode} | {float(np.sqrt(((r - gt_body) ** 2).mean())):.4f} |")
display(Markdown(f"**§3 — {K_ROLL}-step free-run RMSE vs clean GT** (N={N_ROLL}, free-run starts at frame {EF}; "
                 "sampled rollouts reproduce an observation-noise realisation, so compare the two seeds to each "
                 "other and their *structure* to GT rather than their raw RMSE to mean mode)\n\n"
                 "| model | rollout mode | RMSE vs clean GT |\n|---|---|---|\n" + "\n".join(rows)))

# Fig 2 — rollout waterfalls for the primary model (canonical dark/gray spec)
N_CTX = 6
BG, FG, TICK = "#0a0a14", "#a3adc2", "#808a9d"
SHOW = [0, 1]
ctx = test.obs[SHOW][:, EF - N_CTX:EF]
cols = [("GT (sim clean obs)", test.clean_obs[SHOW][:, EF:EF + K_ROLL]),
        ("Mean readout (deterministic)", ROLLS[PRIMARY]["mean"][SHOW]),
        ("Euler sample · fresh noise · seed A", ROLLS[PRIMARY]["sample_fresh A"][SHOW]),
        ("Euler sample · fresh noise · seed B", ROLLS[PRIMARY]["sample_fresh B"][SHOW])]
fig, axes = plt.subplots(len(SHOW), len(cols), figsize=(3.1 * len(cols), 3.4 * len(SHOW)),
                         squeeze=False, facecolor=BG)
for r in range(len(SHOW)):
    for c, (name, body) in enumerate(cols):
        ax = axes[r][c]; ax.set_facecolor(BG)
        ax.imshow(np.clip(np.concatenate([ctx[r], body[r]], axis=0), 0, 1), cmap="gray",
                  vmin=0, vmax=1, aspect="auto", interpolation="nearest")
        ax.axhline(N_CTX - 0.5, color="#fa8850", ls="--", lw=1.3)
        for sp in ax.spines.values(): sp.set_edgecolor(TICK)
        if r == 0: ax.set_title(name, fontsize=8.5, color=FG)
        if c == 0:
            ax.set_ylabel(f"test seq {SHOW[r]}\nsim frame", fontsize=8, color=FG)
            ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
            ax.set_yticklabels([EF - N_CTX, EF, EF + 7, EF + 14], fontsize=7)
        else: ax.set_yticks([])
        ax.set_xlabel("ray", fontsize=8, color=FG); ax.tick_params(colors=TICK, labelsize=7)
fig.suptitle(f"Fig 2 — autoregressive free-run by rollout mode ({PRIMARY})\n{N_CTX} noisy teacher-forced "
             "context frames above the dashed line; below it every column is its own free-run from step 0; "
             "GT column shows the clean render", fontsize=9.5, color=FG)
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(f"{OUT}/fig2_sampled_rollouts.png", dpi=150, facecolor=BG, bbox_inches="tight"); plt.show()

In [ ]:
# [5] §4 The (residual point × diffusion time) probe grid — same protocol as ../DiT/dit_world_state.ipynb,
#     but the states come from the DiT core running over LATENTS.
core = model.core
S = core.cfg.n_sample_steps
N_RESID = core.cfg.n_layers + 1
RESID_LABELS = ["0 · token embedding", "1 · early", "2 · middle", "3 · late", "4 · last (velocity-head input)"]
TAU_KS = [0, 2, 4, 6, 8]
taus_full = torch.linspace(1.0, 0.0, S + 1)
TAU_VALS = [float(taus_full[k]) for k in TAU_KS]
W, Z, D = core.cfg.window, model.latent_dim, core.cfg.d_model

obs_g = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
with torch.no_grad():
    z_seq = model.encode(obs_g)                                   # (B, T, Z) normalised latents
B, T, _ = z_seq.shape
windows, lengths = core._unfold_windows(z_seq)
flat_win = windows.reshape(B * (T - 1), W, Z)
flat_len = lengths.unsqueeze(0).expand(B, -1).reshape(-1)
n_rows = flat_win.shape[0]

STATES = {k: [np.empty((n_rows, D), np.float32) for _ in range(N_RESID)] for k in TAU_KS}
gen = torch.Generator().manual_seed(7)
eps_rows = torch.randn(n_rows, Z, generator=gen)
with torch.no_grad():
    for i0 in range(0, n_rows, 8192):
        sl = slice(i0, min(i0 + 8192, n_rows))
        win, ln = flat_win[sl], flat_len[sl]
        n = win.shape[0]
        cur, nxt = core._window_tokens(win)
        attn = core._window_attn_mask(ln, DEVICE)
        x = eps_rows[sl].to(DEVICE)
        for k in range(S + 1):
            tau_t = torch.zeros(n, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
            nxt_k = torch.cat([nxt[:, :-1], x.unsqueeze(1)], dim=1)
            if k in TAU_KS:
                sink = []
                feats, c = core._trunk(cur, nxt_k, tau_t, attn, resid_sink=sink)
                for L in range(N_RESID):
                    STATES[k][L][sl] = sink[L][:, -1].cpu().numpy()
                v = core.final_layer(feats, c)[:, -1]
            else:
                v = core._denoise(cur, nxt_k, tau_t, attn)[:, -1]
            if k < S:
                x = x + (taus_full[k + 1] - taus_full[k]) * v

P_g = test.positions[:N_PROBE, :T - 1, :N_OBJ].reshape(N_PROBE, T - 1, N_OBJ * 2).astype(np.float32)
vis_g = test.is_visible[:N_PROBE, :T - 1, :N_OBJ].all(axis=2)
lin_r2 = np.zeros((N_RESID, len(TAU_KS))); mlp_r2 = np.zeros_like(lin_r2)
GRID = {}
for j, k in enumerate(TAU_KS):
    for L in range(N_RESID):
        fit = fit_readability_probes(STATES[k][L].reshape(N_PROBE, T - 1, D), P_g, mask=vis_g, device=DEVICE)
        GRID[(L, k)] = fit
        lin_r2[L, j], mlp_r2[L, j] = fit["linear_r2"], fit["mlp_r2"]

hdr = "| residual point | " + " | ".join(f"τ={t:g}" for t in TAU_VALS) + " |\n|---|" + "---|" * len(TAU_VALS) + "\n"
display(Markdown("**§4 — held-out LINEAR position R² by (residual point × τ)** for %s "
                 "(%d/%d train/held-out sequences)\n\n" % (PRIMARY, GRID[(0, 0)]["n_train_seq"],
                                                           GRID[(0, 0)]["n_heldout_seq"])
                 + hdr + "\n".join(f"| {RESID_LABELS[L]} | " +
                                   " | ".join(f"{lin_r2[L, j]:.3f}" for j in range(len(TAU_KS))) + " |"
                                   for L in range(N_RESID))))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for ax, mat, name in [(axes[0], lin_r2, "(a) linear probe"), (axes[1], mlp_r2, "(b) MLP probe")]:
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(TAU_VALS))); ax.set_xticklabels([f"τ={t:g}" for t in TAU_VALS])
    ax.set_yticks(range(N_RESID)); ax.set_yticklabels(RESID_LABELS, fontsize=8)
    for L in range(N_RESID):
        for j in range(len(TAU_VALS)):
            ax.text(j, L, f"{mat[L, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if mat[L, j] < 0.6 else "black")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("diffusion time of the prediction-slot iterate (1 = pure noise, 0 = finished sample)")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Fig 3 — held-out position R² by residual point × diffusion time ({PRIMARY}, states from the "
             "fresh-noise Euler sampler over latents)", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_probe_grid.png", dpi=150, bbox_inches="tight"); plt.show()

## Current results (updated 2026-08-11)

**§1 — the VAE gate passes.** `vae_z16` reconstructs the test split at RMSE **0.1294 vs the noisy input** it was
trained on and **0.0986 vs the clean render** — below the dataset's own noise floor (0.1541), i.e. the 16-d
bottleneck *partially denoises* rather than losing structure (accepted design choice). `vae_z8`: 0.1380 / 0.0924.
Single-frame position readout from the 16-d code (linear 0.229 / MLP 0.540) matches the raw 128-d observation
(0.261 / 0.515): **an 8× compression costs essentially nothing about where the objects are.**

**§2 — predictive quality matches pixel space.** Decoded next-step MSE for **Latent DiT · z=16 · window 4** is
**0.02517 vs noisy** (pixel DiT W4 0.02443, GRU 0.02362) — but its **VAE floor is 0.01672**, so its dynamics
contribute only +0.00846 of excess error. On the fairer clean-target metric it is **0.01174 vs the pixel DiT's
0.01186**, i.e. *marginally better structure prediction than the pixel model*. Window 2 is worse than window 4
(0.02614 / 0.01284) and z=8 sits between (0.02552 / 0.01215).

**§3 — multi-step sampling is stable.** 15-step free-runs: mean 0.2168, `sample_fresh` seeds 0.2411 / 0.2322
(z=16·W4). Fig 2 shows coherent object bands with no collapse across independent seeds; the sampled rollouts
are visibly *less* speckled than the pixel DiT's because the bottleneck cannot encode per-ray noise.

**§4 — the state is highly readable, and flat in τ.** Linear position R² rises with depth to **0.807–0.810** at
the late/last residual points (pixel DiT: 0.70) and is **flat across τ** (τ=1 → τ=0 within ±0.02), replicating
the pixel-DiT finding that the position belief is carried by the **context pathway**, not the iterate. Token
embedding reads only 0.22 linearly.

## Summary (interpretation — clearly marked as such)

The latent DiT passes every gate: a VAE that keeps the world state while denoising, next-step structure
prediction at or slightly better than the pixel DiT once its floor is accounted for, a stable stochastic
sampler, and the most position-readable state of any model in this thread. The flat-in-τ readability map
reproduces the pixel-space result — the belief lives in the clean context, and the diffusion iterate adds
almost nothing until it is finished. That combination (higher readability, same context-carried belief) is what
makes this architecture the sharpest available test of readable≠controllable; the editors are in
`../input_grad_steering/input_grad_steering_latent_dit.ipynb`.